# 01 — Pipeline NLP : Extraction PDF & Chunking

Ce notebook démontre et évalue le pipeline NLP :
1. Extraction de texte depuis un PDF avec PyMuPDF
2. Détection des chapitres par analyse de police
3. Découpage en chunks avec chevauchement
4. Préservation des métadonnées (page, chapitre, index)

In [ ]:
import sys
sys.path.insert(0, '..')

from backend.services.pdf_extractor import PDFExtractor
from backend.services.chunker import TextChunker
import json

## 1. Extraction PDF

In [ ]:
# Remplacez par le chemin de votre PDF de test
PDF_PATH = "../data/sample_courses/test_course.pdf"

extractor = PDFExtractor()
result = extractor.extract(PDF_PATH)

print(f"Pages extraites : {result['total_pages']}")
print(f"Chapitres détectés : {len(result['chapters'])}")
for ch in result['chapters']:
    print(f"  - {ch['title']} (page {ch['start_page']})")

# Aperçu du texte
print(f"\nLongueur du texte brut : {len(result['full_text'])} caractères")
print(result['full_text'][:500])

## 2. Découpage en Chunks

In [ ]:
chunker = TextChunker(chunk_size=400, overlap=50)
chunks = chunker.split(result['pages'])

print(f"Nombre de chunks : {len(chunks)}")
print(f"\n--- Chunk 0 ---")
print(json.dumps(chunks[0], ensure_ascii=False, indent=2)[:600])

## 3. Analyse statistique des chunks

In [ ]:
import statistics

lengths = [len(c['text'].split()) for c in chunks]
print(f"Taille min  : {min(lengths)} mots")
print(f"Taille max  : {max(lengths)} mots")
print(f"Taille moy. : {statistics.mean(lengths):.1f} mots")
print(f"Écart-type  : {statistics.stdev(lengths):.1f} mots")
print(f"\nChapitres couverts : {len(set(c.get('chapter', 'N/A') for c in chunks))}")

## 4. Vérification du chevauchement (overlap)

In [ ]:
if len(chunks) >= 2:
    c0_words = chunks[0]['text'].split()
    c1_words = chunks[1]['text'].split()
    
    # Trouver le chevauchement
    overlap_words = []
    for i in range(min(100, len(c0_words))):
        suffix = ' '.join(c0_words[-(i+1):])
        prefix = ' '.join(c1_words[:i+1])
        if suffix == prefix:
            overlap_words = c0_words[-(i+1):]

    print(f"Mots en commun entre chunk 0 et 1 : ~{len(overlap_words)}")
    print(f"Overlap attendu : ~50 mots")
else:
    print("Pas assez de chunks pour vérifier l'overlap")

## Conclusion

Le pipeline NLP :
- Extrait correctement le texte structuré du PDF
- Détecte les titres de chapitres par analyse typographique
- Produit des chunks de taille contrôlée (~400 mots) avec overlap (~50 mots)
- Préserve les métadonnées (page, chapitre) pour la traçabilité